# AgentHire Extractor Training (SmolLM2 + Unsloth + Kaggle)

This notebook builds a highly specialized JSON extraction model for the AgentHire pipeline.

**Phase 1:** Uses a local `llama3.2` (via Ollama) as a Teacher Model to annotate raw resumes from Kaggle.

**Phase 2:** Fine-tunes the fast `SmolLM2-360M` model on this annotated data using Unsloth.

**Phase 3:** Exports the final model to GGUF format for production use.

In [ ]:
!pip install unsloth trl datasets transformers accelerate kagglehub pandas langchain-ollama

## Part 1: Dataset Generation (Kaggle + Teacher LLM)
Ensure Ollama is running locally with `llama3.2` installed (`ollama pull llama3.2`).

In [ ]:
import json
import pandas as pd
from pathlib import Path
import kagglehub
from kagglehub import KaggleDatasetAdapter
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

# --- Configuration ---
DATASET_SIZE = 100 # Start small to test the pipeline, then increase to 1000+
OUTPUT_FILE = "data/resume_dataset.json"
TEACHER_MODEL = "llama3.2"

SYSTEM_PROMPT = """You are a precise document parsing agent.
Your ONLY job is to read the provided document text and return a single valid JSON object.
Rules:
- Return ONLY the JSON object. No explanation, markdown, or text outside the JSON.
- If a field is not present in the document, set it to null.
- For list fields (skills, experience, education), use an empty list [] if nothing is found.
- Skills must be individual strings, not comma-joined.

Output schema:
{
  \"name\": string or null,
  \"email\": string or null,
  \"phone\": string or null,
  \"website\": string or null,
  \"skills\": [string, ...],
  \"experience\": [{\"title\": string, \"company\": string, \"duration\": string}, ...],
  \"education\": [{\"degree\": string, \"institution\": string, \"year\": string}, ...]
}"""

def load_kaggle_resumes(num_samples: int) -> list[str]:
    print("Downloading dataset via kagglehub...")
    df = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "snehaanbhawal/resume-dataset",
        "Resume.csv",
    )
    print(f"Dataset loaded. Total records: {len(df)}")
    df_sampled = df.sample(n=num_samples, random_state=42)
    return df_sampled['Resume_str'].tolist()

def generate_training_data() -> list[dict]:
    raw_resumes = load_kaggle_resumes(DATASET_SIZE)
    teacher_llm = OllamaLLM(model=TEACHER_MODEL, temperature=0.0, format="json")
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "{text}")
    ])
    chain = prompt_template | teacher_llm

    training_data = []
    print(f"Generating ground truth for {DATASET_SIZE} resumes using {TEACHER_MODEL}...")
    for i, resume_text in enumerate(raw_resumes):
        if i % 10 == 0:
            print(f"Processing {i}/{DATASET_SIZE}...")

        clean_text = str(resume_text).replace("\r", "").strip()[:4000]

        try:
            ground_truth_json = chain.invoke({"text": clean_text})
            json.loads(ground_truth_json)  # Validate JSON

            training_data.append({
                "instruction": SYSTEM_PROMPT,
                "input": clean_text,
                "output": ground_truth_json
            })
        except Exception as e:
            print(f"Skipping index {i} due to parsing error: {e}")

    return training_data

output_path = Path(OUTPUT_FILE)
output_path.parent.mkdir(parents=True, exist_ok=True)

dataset_records = generate_training_data()

with output_path.open("w", encoding="utf-8") as f:
    json.dump(dataset_records, f, indent=2)

print(f"\nSuccessfully annotated {len(dataset_records)} examples!")

## Part 2: Fine-Tuning SmolLM2-360M with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"
MAX_SEQ_LENGTH = 1024
DATASET_PATH = Path("data/resume_dataset.json")
HF_OUTPUT_DIR = "./agenthire-smollm-extractor"
GGUF_OUTPUT_DIR = "agenthire-extractor-gguf"

with DATASET_PATH.open(encoding="utf-8") as f:
    raw = json.load(f)

def format_example(ex):
    return {
        "text": (
            f"### System:\n{ex['instruction']}\n\n"
            f"### Document:\n{ex['input']}\n\n"
            f"### Extracted JSON:\n{ex['output']}"
        )
    }

hf_dataset = Dataset.from_list(raw).map(format_example, remove_columns=list(raw[0].keys()))
print("Dataset formatted for training.")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

In [ ]:
cuda_available = torch.cuda.is_available()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=cuda_available and not torch.cuda.is_bf16_supported(),
        bf16=cuda_available and torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir="./outputs",
        save_strategy="epoch",
        report_to="none",
    ),
)

trainer.train()

## Part 3: Export and Save

In [ ]:
# --- Local GGUF Export (For Ollama) ---
model.save_pretrained(HF_OUTPUT_DIR)
tokenizer.save_pretrained(HF_OUTPUT_DIR)

model.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

print("Done! GGUF model exported successfully.")

# --- Save to Hugging Face directly ---
HF_REPO = "nimendraai/SmolLM2-360M-AgentHire-Extractor"

print("Pushing standard weights to Hub...")
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print("Pushing GGUF to Hub...")
model.push_to_hub_gguf(
    HF_REPO,
    tokenizer,
    quantization_method="q4_k_m",
)
print("Done! All formats uploaded.")